# 024 — SQL-RAG (Structured Data Retrieval)
**Advanced RAG Series** | Query relational databases with natural language

Covers: NL-to-SQL · SQLite with LangChain · Schema-aware prompting · Error recovery


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
# ── 1. Inspect the existing SQLite database ─────────────────────────────
import sqlite3, os

DB_PATH = os.path.join(os.getcwd(), '..', 'datasets', 'ai_papers.db')
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# List tables
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [r[0] for r in cur.fetchall()]
print("Tables:", tables)

# Schema for each table
for table in tables:
    cur.execute(f"PRAGMA table_info({table})")
    cols = cur.fetchall()
    print(f"\n{table}:")
    for col in cols:
        print(f"  {col['name']:<20} {col['type']}")


Tables: ['ai_papers']

ai_papers:
  id                   INTEGER
  title                TEXT
  authors              TEXT
  year                 INTEGER
  venue                TEXT
  citations            INTEGER
  topic                TEXT
  abstract             TEXT


In [4]:
# ── 2. Sample data ──────────────────────────────────────────────────────
for table in tables:
    cur.execute(f"SELECT COUNT(*) as cnt FROM {table}")
    count = cur.fetchone()['cnt']
    print(f"{table}: {count} rows")
    cur.execute(f"SELECT * FROM {table} LIMIT 3")
    rows = cur.fetchall()
    for row in rows:
        print(dict(row))
    print()


ai_papers: 10 rows
{'id': 1, 'title': 'Attention Is All You Need', 'authors': 'Vaswani et al.', 'year': 2017, 'venue': 'NeurIPS', 'citations': 80000, 'topic': 'Transformers', 'abstract': 'We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.'}
{'id': 2, 'title': 'BERT: Pre-training of Deep Bidirectional Transformers', 'authors': 'Devlin et al.', 'year': 2019, 'venue': 'NAACL', 'citations': 55000, 'topic': 'Pre-training', 'abstract': 'We introduce BERT, which stands for Bidirectional Encoder Representations from Transformers. BERT is designed to pre-train deep bidirectional representations from unlabelled text.'}
{'id': 3, 'title': 'GPT-3: Language Models are Few-Shot Learners', 'authors': 'Brown et al.', 'year': 2020, 'venue': 'NeurIPS', 'citations': 35000, 'topic': 'Large Language Models', 'abstract': 'We demonstrate that scaling language models greatly improves task-agnostic few-shot 

In [5]:
# ── 3. Schema-aware SQL generation ──────────────────────────────────────
def get_schema() -> str:
    schema_parts = []
    for table in tables:
        cur.execute(f"PRAGMA table_info({table})")
        cols = cur.fetchall()
        col_defs = ", ".join([f"{c['name']} {c['type']}" for c in cols])
        schema_parts.append(f"Table {table}({col_defs})")
    return "\n".join(schema_parts)

SQL_GEN_PROMPT = (
    "You are a SQL expert. Given a SQLite database schema and a question, "
    "write a single valid SQLite SELECT query to answer it.\n"
    "Rules:\n"
    "- Output ONLY the SQL query, no explanation, no markdown\n"
    "- Use only SELECT statements\n"
    "- Use exact table and column names from the schema\n\n"
    "Schema:\n{schema}\n\n"
    "Question: {question}\n\n"
    "SQL:"
)

schema = get_schema()
print("Schema sent to LLM:")
print(schema)


Schema sent to LLM:
Table ai_papers(id INTEGER, title TEXT, authors TEXT, year INTEGER, venue TEXT, citations INTEGER, topic TEXT, abstract TEXT)


In [6]:
# ── 4. Execute NL-to-SQL safely ─────────────────────────────────────────
def nl_to_sql(question: str) -> str:
    prompt = SQL_GEN_PROMPT.format(schema=schema, question=question)
    response = llm.invoke(prompt)
    sql = response.content.strip()
    # Strip markdown fences
    if sql.startswith("```"):
        sql = sql.split("```")[1]
        if sql.lower().startswith("sql"):
            sql = sql[3:]
    return sql.strip()

def safe_execute(sql: str) -> list[dict]:
    # Reject anything that's not a SELECT
    clean = sql.upper().strip()
    if not clean.startswith("SELECT"):
        raise ValueError(f"Only SELECT allowed. Got: {sql[:50]}")
    for blocked in ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "CREATE", "TRUNCATE"]:
        if blocked in clean:
            raise ValueError(f"Blocked keyword: {blocked}")
    cur.execute(sql)
    return [dict(row) for row in cur.fetchall()]

question = "How many papers are in each category?"
sql = nl_to_sql(question)
print(f"Question: {question}")
print(f"Generated SQL: {sql}")
rows = safe_execute(sql)
print(f"Result ({len(rows)} rows):")
for row in rows[:5]:
    print(" ", row)


Question: How many papers are in each category?
Generated SQL: SELECT topic, COUNT(*) FROM ai_papers GROUP BY topic
Result (9 rows):
  {'topic': 'Advanced RAG', 'COUNT(*)': 1}
  {'topic': 'LLM Frameworks', 'COUNT(*)': 2}
  {'topic': 'Large Language Models', 'COUNT(*)': 1}
  {'topic': 'Multimodal', 'COUNT(*)': 1}
  {'topic': 'Pre-training', 'COUNT(*)': 1}


In [7]:
# ── 5. Full SQL-RAG pipeline ────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate

ANSWER_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful data analyst.\n"
    "Question: {question}\n"
    "SQL query used: {sql}\n"
    "Query results: {results}\n\n"
    "Answer the question naturally based on the results:"
)

def sql_rag(question: str) -> dict:
    sql = nl_to_sql(question)
    try:
        rows = safe_execute(sql)
        results_str = str(rows[:20])
        chain = ANSWER_PROMPT | llm
        answer = chain.invoke({
            "question": question,
            "sql": sql,
            "results": results_str,
        }).content
        return {"sql": sql, "rows": rows, "answer": answer}
    except Exception as e:
        return {"sql": sql, "error": str(e)}

result = sql_rag("Which year had the most published papers?")
print(f"SQL: {result['sql']}")
print(f"\nAnswer: {result['answer']}")


SQL: SELECT year FROM ai_papers GROUP BY year ORDER BY COUNT(id) DESC LIMIT 1

Answer: Based on the query results, it appears that the year with the most published papers is 2022. This suggests that there was a significant increase in research and publication activity in the field of AI in that year, possibly due to advancements in technology, increased funding, or growing interest in the field.


In [8]:
# ── 6. SQL-RAG with error recovery ──────────────────────────────────────
REPAIR_PROMPT = (
    "This SQLite query failed:\n{sql}\n"
    "Error: {error}\n"
    "Schema: {schema}\n"
    "Fix the query. Output ONLY the corrected SQL, no explanation."
)

def sql_rag_with_recovery(question: str, max_retries: int = 2) -> dict:
    sql = nl_to_sql(question)
    for attempt in range(max_retries + 1):
        try:
            rows = safe_execute(sql)
            chain = ANSWER_PROMPT | llm
            answer = chain.invoke({
                "question": question, "sql": sql, "results": str(rows[:20])
            }).content
            return {"sql": sql, "rows": rows, "answer": answer, "attempts": attempt + 1}
        except Exception as e:
            if attempt == max_retries:
                return {"sql": sql, "error": str(e), "attempts": attempt + 1}
            # Repair
            repair_prompt = REPAIR_PROMPT.format(sql=sql, error=str(e), schema=schema)
            sql = llm.invoke(repair_prompt).content.strip()

result = sql_rag_with_recovery("List the top 5 most cited papers")
print(f"SQL: {result['sql']}")
print(f"Attempts: {result.get('attempts', 1)}")
print(f"\nAnswer: {result.get('answer', result.get('error'))}")


SQL: SELECT title, citations FROM ai_papers ORDER BY citations DESC LIMIT 5
Attempts: 1

Answer: Based on the query results, the top 5 most cited papers in the field of AI are:

1. **"Attention Is All You Need"** with 80,000 citations - This paper revolutionized the field of natural language processing by introducing the Transformer architecture, which has since become a cornerstone of many AI models.

2. **"BERT: Pre-training of Deep Bidirectional Transformers"** with 55,000 citations - This paper introduced the BERT model, which has achieved state-of-the-art results in many NLP tasks and has been widely adopted in industry and academia.

3. **"GPT-3: Language Models are Few-Shot Learners"** with 35,000 citations - This paper introduced the GPT-3 model, which has demonstrated impressive language understanding and generation capabilities, and has been used in a variety of applications, including chatbots and language translation.

4. **"CLIP: Learning Transferable Visual Models from Na

In [9]:
# ── 7. When to use SQL-RAG ──────────────────────────────────────────────
conn.close()

print("SQL-RAG is the right choice when:")
print("  Data is STRUCTURED (tables, rows, columns)")
print("  Questions need aggregation: count, sum, average, group by")
print("  Questions need filtering: 'papers after 2020 with >100 citations'")
print("  Data updates frequently (SQL DB is single source of truth)")
print()
print("Use vector RAG when:")
print("  Data is UNSTRUCTURED (documents, PDFs, text files)")
print("  Questions are conceptual: 'explain', 'summarise', 'compare'")
print("  Exact keyword or semantic matching is needed")
print()
print("Combine both (Hybrid SQL + Vector RAG) when:")
print("  System has both structured metadata AND unstructured content")
print("  e.g. paper DB: filter by year/author with SQL, read abstract with RAG")


SQL-RAG is the right choice when:
  Data is STRUCTURED (tables, rows, columns)
  Questions need aggregation: count, sum, average, group by
  Questions need filtering: 'papers after 2020 with >100 citations'
  Data updates frequently (SQL DB is single source of truth)

Use vector RAG when:
  Data is UNSTRUCTURED (documents, PDFs, text files)
  Questions are conceptual: 'explain', 'summarise', 'compare'
  Exact keyword or semantic matching is needed

Combine both (Hybrid SQL + Vector RAG) when:
  System has both structured metadata AND unstructured content
  e.g. paper DB: filter by year/author with SQL, read abstract with RAG
